
## File 01 — Job Parameters and Task Configuration:


This notebook prepares your pipeline to run as a parameterized Databricks Job instead of relying on hardcoded values.

It covers:

- Databricks widgets.
- Job parameters.
- Pipeline run IDs.
- Dataset selection.
- Layer selection.
- Execution mode.
- Parameter validation.
- Job configuration output.


## Example A — Full Pipeline

Execute all datasets across all pipeline layers using incremental processing.

```python
dataset_name = "all"
pipeline_layer = "ALL"
execution_mode = "INCREMENTAL"
```

**Execution:**
- Process all seven datasets.
- Execute Bronze, Silver, and Gold layers.
- Process only new or changed data.
- Update target Delta tables incrementally.


## Example B — Only Customer Bronze Ingestion

Execute Bronze ingestion for the customers dataset only.

```python
dataset_name = "customers"
pipeline_layer = "BRONZE"
execution_mode = "INCREMENTAL"
```

**Execution:**
- Process only customer source files.
- Use Auto Loader to discover new files.
- Apply checkpoint-based ingestion.
- Write new records into the Bronze customers table.
- Do not execute Silver or Gold tasks.


## Example C — Full Gold Refresh

Rebuild all Gold analytics tables using the available validated Silver data.

```python
dataset_name = "all"
pipeline_layer = "GOLD"
execution_mode = "FULL_REFRESH"
```

**Execution:**
- Process all Gold analytics datasets.
- Recalculate aggregations and business metrics.
- Rebuild Gold target tables.
- Refresh dashboard-ready datasets.

**Important:** FULL_REFRESH should replace or rebuild the Gold targets safely. It should not unnecessarily reload Bronze or Silver data.


## Example D — Validation Only

Execute validation checks across all pipeline layers without modifying business data.

```python
dataset_name = "all"
pipeline_layer = "ALL"
execution_mode = "VALIDATION_ONLY"
```

**Execution:**
- Validate Bronze tables and ingestion metadata.
- Validate Silver business keys and data quality.
- Check processing ledger and quarantine records.
- Validate Delta MERGE reports.
- Validate Gold analytics tables and record counts.
- Generate a consolidated validation report.

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid
import json


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

VOLUME_SCHEMA = "default"
VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/"
    f"{VOLUME}/ecommerce_lakehouse"
)

LANDING_PATH = f"{BASE_PATH}/landing"

CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"

SCHEMA_PATH = f"{BASE_PATH}/schemas"

LOG_PATH = f"{BASE_PATH}/logs"


# ============================================================
# 3. CREATE JOB PARAMETERS
# ============================================================

# Remove existing widgets before creating them.

dbutils.widgets.removeAll()


# Pipeline name

dbutils.widgets.text(
    "pipeline_name",
    "ecommerce_lakehouse_pipeline"
)


# Dataset selection

dbutils.widgets.dropdown(
    "dataset_name",
    "all",
    [
        "all",
        "customers",
        "products",
        "orders",
        "order_items",
        "payments",
        "shipments",
        "inventory_snapshots"
    ]
)


# Pipeline layer

dbutils.widgets.dropdown(
    "pipeline_layer",
    "ALL",
    [
        "ALL",
        "BRONZE",
        "SILVER",
        "GOLD"
    ]
)


# Execution mode

dbutils.widgets.dropdown(
    "execution_mode",
    "INCREMENTAL",
    [
        "INCREMENTAL",
        "FULL_REFRESH",
        "VALIDATION_ONLY"
    ]
)


# Run ID

dbutils.widgets.text(
    "run_id",
    ""
)


# Enable data quality checks

dbutils.widgets.dropdown(
    "enable_data_quality",
    "true",
    ["true", "false"]
)


# Enable referential integrity checks

dbutils.widgets.dropdown(
    "enable_referential_integrity",
    "true",
    ["true", "false"]
)


# Fail pipeline if validation fails

dbutils.widgets.dropdown(
    "fail_on_validation_error",
    "true",
    ["true", "false"]
)


# ============================================================
# 4. READ JOB PARAMETERS
# ============================================================

PIPELINE_NAME = dbutils.widgets.get(
    "pipeline_name"
).strip()

DATASET_NAME = dbutils.widgets.get(
    "dataset_name"
).strip().lower()

PIPELINE_LAYER = dbutils.widgets.get(
    "pipeline_layer"
).strip().upper()

EXECUTION_MODE = dbutils.widgets.get(
    "execution_mode"
).strip().upper()

JOB_RUN_ID = dbutils.widgets.get(
    "run_id"
).strip()

ENABLE_DATA_QUALITY = (
    dbutils.widgets.get("enable_data_quality").lower()
    == "true"
)

ENABLE_REFERENTIAL_INTEGRITY = (
    dbutils.widgets.get(
        "enable_referential_integrity"
    ).lower()
    == "true"
)

FAIL_ON_VALIDATION_ERROR = (
    dbutils.widgets.get(
        "fail_on_validation_error"
    ).lower()
    == "true"
)


# ============================================================
# 5. GENERATE RUN ID IF NOT PROVIDED
# ============================================================

if not JOB_RUN_ID:

    JOB_RUN_ID = str(uuid.uuid4())


# ============================================================
# 6. VALIDATE JOB PARAMETERS
# ============================================================

VALID_DATASETS = [

    "all",
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"

]

VALID_LAYERS = [

    "ALL",
    "BRONZE",
    "SILVER",
    "GOLD"

]

VALID_EXECUTION_MODES = [

    "INCREMENTAL",
    "FULL_REFRESH",
    "VALIDATION_ONLY"

]


def validate_job_parameters():

    if not PIPELINE_NAME:

        raise ValueError(
            "Pipeline name cannot be empty."
        )

    if DATASET_NAME not in VALID_DATASETS:

        raise ValueError(
            f"Invalid dataset: {DATASET_NAME}"
        )

    if PIPELINE_LAYER not in VALID_LAYERS:

        raise ValueError(
            f"Invalid pipeline layer: {PIPELINE_LAYER}"
        )

    if EXECUTION_MODE not in VALID_EXECUTION_MODES:

        raise ValueError(
            f"Invalid execution mode: {EXECUTION_MODE}"
        )

    if not JOB_RUN_ID:

        raise ValueError(
            "Job run ID cannot be empty."
        )

    return True


validate_job_parameters()


# ============================================================
# 7. BUILD DATASET EXECUTION LIST
# ============================================================

ALL_DATASETS = [

    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"

]


if DATASET_NAME == "all":

    SELECTED_DATASETS = ALL_DATASETS

else:

    SELECTED_DATASETS = [DATASET_NAME]


# ============================================================
# 8. BUILD PIPELINE CONFIGURATION
# ============================================================

PIPELINE_CONFIG = {

    "pipeline_name": PIPELINE_NAME,

    "run_id": JOB_RUN_ID,

    "dataset_name": DATASET_NAME,

    "selected_datasets": SELECTED_DATASETS,

    "pipeline_layer": PIPELINE_LAYER,

    "execution_mode": EXECUTION_MODE,

    "enable_data_quality": ENABLE_DATA_QUALITY,

    "enable_referential_integrity":
        ENABLE_REFERENTIAL_INTEGRITY,

    "fail_on_validation_error":
        FAIL_ON_VALIDATION_ERROR,

    "catalog": CATALOG,

    "bronze_schema": BRONZE_SCHEMA,

    "silver_schema": SILVER_SCHEMA,

    "gold_schema": GOLD_SCHEMA,

    "base_path": BASE_PATH,

    "landing_path": LANDING_PATH,

    "checkpoint_path": CHECKPOINT_PATH,

    "schema_path": SCHEMA_PATH,

    "log_path": LOG_PATH,

    "created_at": datetime.now(
        timezone.utc
    ).isoformat()

}


# ============================================================
# 9. VALIDATE REQUIRED TABLES
# ============================================================

def check_table_exists(table_name):

    return spark.catalog.tableExists(
        table_name
    )


def validate_layer_tables(layer):

    if layer == "BRONZE":

        schema_name = BRONZE_SCHEMA

        prefix = ""

    elif layer == "SILVER":

        schema_name = SILVER_SCHEMA

        prefix = "silver_"

    elif layer == "GOLD":

        return True

    else:

        return True

    results = []

    for dataset in SELECTED_DATASETS:

        if dataset == "all":

            continue

        table_name = (
            f"{CATALOG}.{schema_name}."
            f"{prefix}{dataset}"
        )

        exists = check_table_exists(
            table_name
        )

        results.append({

            "table_name": table_name,

            "exists": exists

        })

    return results


# ============================================================
# 10. DISPLAY SELECTED JOB CONFIGURATION
# ============================================================

print("\n" + "=" * 70)

print("DATABRICKS JOB CONFIGURATION")

print("=" * 70)

print(
    json.dumps(
        PIPELINE_CONFIG,
        indent=4
    )
)


# ============================================================
# 11. DISPLAY SELECTED DATASETS
# ============================================================

print("\nSELECTED DATASETS")

for dataset in SELECTED_DATASETS:

    print(f"- {dataset}")


# ============================================================
# 12. DISPLAY TABLE VALIDATION
# ============================================================

if PIPELINE_LAYER in ["BRONZE", "SILVER"]:

    table_validation = validate_layer_tables(
        PIPELINE_LAYER
    )

    print("\nTABLE VALIDATION")

    for result in table_validation:

        print(
            f"{result['table_name']} : "
            f"{'EXISTS' if result['exists'] else 'MISSING'}"
        )


# ============================================================
# 13. DISPLAY EXECUTION SUMMARY
# ============================================================

print("\n" + "=" * 70)

print("JOB PARAMETER VALIDATION SUMMARY")

print("=" * 70)

print(f"Pipeline: {PIPELINE_NAME}")

print(f"Run ID: {JOB_RUN_ID}")

print(f"Dataset: {DATASET_NAME}")

print(f"Layer: {PIPELINE_LAYER}")

print(f"Execution mode: {EXECUTION_MODE}")

print(
    f"Data quality enabled: {ENABLE_DATA_QUALITY}"
)

print(
    "Referential integrity enabled: "
    f"{ENABLE_REFERENTIAL_INTEGRITY}"
)

print(
    f"Fail on validation error: "
    f"{FAIL_ON_VALIDATION_ERROR}"
)

print("=" * 70)

print("JOB PARAMETER CONFIGURATION COMPLETED.")


# ============================================================
# 14. PASS CONFIGURATION TO NEXT TASKS
# ============================================================

# Databricks Jobs tasks run in separate execution contexts.
# Python variables do not automatically transfer between tasks.
#
# Configure the same parameters in dependent tasks using
# Job parameters or task parameters.
#
# Example task parameter:
#
# {
#     "pipeline_name": "{{job.parameters.pipeline_name}}",
#     "run_id": "{{job.run_id}}",
#     "dataset_name": "{{job.parameters.dataset_name}}",
#     "pipeline_layer": "{{job.parameters.pipeline_layer}}",
#     "execution_mode": "{{job.parameters.execution_mode}}"
# }
#
# These are Databricks dynamic value references.
# Configure them in the Job task parameter settings.
